# 9. How deep should the tree grow?

**Would asking more questions help the tree predict MSC for a new farm?** Try one validation round, then repeat it to choose how deep the tree can grow. The 21 final test farms are reserved for Chapter 10.

**Open.** Restart R and save the supplied script as `scripts/my-09-choose-tree.R`. This chapter uses `outputs/checkpoints/tree-input.rds` from Chapter 8. Run the named blocks in order.

## 9.1 How many questions can a tree ask?

Follow one path from the first question to a prediction. Count the questions along the longest path in each diagram. This count is the tree’s **depth**.

```mermaid
flowchart TB
    accTitle: Count Questions Along A Path
    accDescr: In the depth-one tree, each path asks one question. In the depth-two tree, the longest path asks two questions, while one path reaches a prediction after just one.

    subgraph depth_one ["Depth 1"]
        first_shallow{"Question 1"}
        first_shallow -->|Yes| shallow_yes(["Prediction"])
        first_shallow -->|No| shallow_no(["Prediction"])
    end

    subgraph depth_two ["Depth 2"]
        first_deeper{"Question 1"}
        first_deeper -->|Yes| second_deeper{"Question 2"}
        first_deeper -->|No| deeper_no(["Prediction"])
        second_deeper -->|Yes| second_yes(["Prediction"])
        second_deeper -->|No| second_no(["Prediction"])
    end
```

When fitting a tree, **maximum depth** sets the number of questions it can ask along a path. A path can end sooner, as the first “No” branch of the depth-2 tree shows.

**Write your guess.** If we allow up to three questions along a path, will the tree’s error go up or down on the records it learned from? What about on farms it has yet to see? Save both guesses for Section 9.4.

Run {span .script-block}`BLOCK · load-rpart-for-validation` and {span .script-block}`BLOCK · load-tree-input` to load the package and study data.

In [1]:
library(rpart)

{span .script-block}`BLOCK · load-tree-input`

In [2]:
source("scripts/tree_helpers.R")
if (!file.exists("outputs/checkpoints/tree-input.rds")) stop("Run Chapter 8 first to create tree-input.rds.")
tree_input <- readRDS("outputs/checkpoints/tree-input.rds")
train_survey <- tree_input$train

Run {span .script-block}`BLOCK · depth-candidates`. We will try maximum depths 1, 2 and 3.

In [3]:
candidate_depths <- 1:3
candidate_depths

[1] 1 2 3

## 9.2 Follow group 1 through one validation round

**Study data.** For this round, use a maximum depth of **1** and set aside **group 1**. The other four groups fit the tree. Group 1 gives us farms the tree has yet to see, so we can check how well it predicts their lab results. This check is called **validation**.

```mermaid
flowchart TB
    accTitle: Whole Farms And One Validation Round
    accDescr: The 102 farms are split into 81 training and 21 final test farms. In round one, groups 2 through 5 fit the medians and tree. Group 1 checks the resulting predictions. The 21 final test farms are reserved for Chapter 10.

    all_farms["102 farms / 525 records"] --> training_farms["81 training farms / 420 records"]
    all_farms --> final_test["21 final test farms / 105 records<br/>Chapter 10"]
    training_farms --> fitting_farms["Groups 2–5<br/>64 farms / 336 records"]
    training_farms --> validation_farms["Group 1<br/>17 farms / 84 records"]
    fitting_farms -->|Learn| round_model["Round 1 medians and tree"]
    round_model -->|Predict| group_predictions["84 predictions to check"]
    validation_farms -->|Supply inputs| group_predictions
```

### 1. Fit with groups 2–5

Run {span .script-block}`BLOCK · training-folds` to assign the **81 training farms** to five groups, called **folds**. Every record from a farm receives that farm’s fold number.

In [4]:
# Fix the algorithms as in Chapter 6, then set the starting point for this shuffle.
RNGkind(kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
set.seed(2027)
training_farms <- sort(unique(train_survey$FarmID), method = "radix")
# Repeat the labels 1 to 5 until there is one per farm, then shuffle them.
fold_labels <- sample(rep(1:5, length.out = length(training_farms)))
fold_table <- data.frame(FarmID = training_farms, fold = fold_labels)
# match() finds each record's farm in fold_table so we can look up its fold number.
row_fold <- fold_table$fold[match(train_survey$FarmID, fold_table$FarmID)]
stopifnot(!any(fold_table$FarmID %in% tree_input$test$FarmID))
data.frame(fold = 1:5, validation_farms = as.integer(table(fold_labels)),
           validation_records = as.integer(table(row_fold)))

fold,validation_farms,validation_records
<int>,<int>,<int>
1,17,84
2,16,86
3,16,86
4,16,83
5,16,81


Run {span .script-block}`BLOCK · one-round-data`. Find the **336 records from 64 farms** for fitting and the **84 records from 17 farms** for validation.

In [5]:
fitting <- train_survey[row_fold != 1, ]
validation <- train_survey[row_fold == 1, ]
data.frame(role = c("Fit", "Validate"), records = c(nrow(fitting), nrow(validation)),
           farms = c(length(unique(fitting$FarmID)), length(unique(validation$FarmID))))

role,records,farms
<chr>,<int>,<int>
Fit,336,64
Validate,84,17


Learn the medians from the fitting records. Use those medians for missing survey inputs in both groups.

```{admonition} Invented data — learn the median from fitting farms
:class: note dropdown

If the fitting farms have 30, 40 and 200 cows, their median is 40. A hidden farm with a missing cow count receives 40 in the model-input copy.
```

**Back to the study data.** Run {span .script-block}`BLOCK · one-round-prepare`. The cow median is **77.5**. It comes from this round’s fitting records. The median of **72** in Chapter 8 came from all 420 training records.

In [6]:
# Same median rule as Chapter 8, now learned only from this round's fitting records.
round_medians <- learn_medians(fitting)
fitting_ready <- fill_predictors(fitting, round_medians)
validation_ready <- fill_predictors(validation, round_medians)
data.frame(feature = tree_features, fitting_median = unname(round_medians))

feature,fitting_median
<chr>,<dbl>
pasture_hours,9.0
milking_cows,77.5


Run {span .script-block}`BLOCK · one-round-fit`. Find `data = fitting_ready` and `maxdepth <- 1` in the code. These show which records fit this tree and how many questions it can ask along a path.

In [7]:
round_settings <- fixed_tree_settings
round_settings$maxdepth <- 1
round_tree <- rpart(log10_msc ~ pasture_hours + milking_cows,
                    data = fitting_ready, method = "anova", control = round_settings)

### 2. Predict group 1

Run {span .script-block}`BLOCK · one-round-predict` to predict all 84 records in group 1. The output shows the first record so you can follow it into the next section.

In [8]:
round_predictions <- predict(round_tree, newdata = validation_ready)
# Show the first record with the two labels that identify it.
round_one_record <- data.frame(
  FarmID = validation_ready$FarmID[1],
  Sampling = validation_ready$Sampling[1],
  observed_log10 = validation_ready$log10_msc[1],
  predicted_log10 = as.numeric(round_predictions[1]))
round_one_record

FarmID,Sampling,observed_log10,predicted_log10
<chr>,<chr>,<dbl>,<dbl>
R4,A,0.544068,0.6842116


Write down the two labels **R4** and **A**. Together they identify farm R4’s record from sampling round A. Its prediction is about **0.6842** on the log10 scale.

### 3. Check group 1’s predictions

Compare R4’s lab result, about **0.5441**, with its prediction. Use [Chapter 7’s subtraction](07-baseline.ipynb#id-7-2-check-one-prediction-by-hand). The prediction was about **0.1401** above the observed value on the log10 scale.

Now run {span .script-block}`BLOCK · one-round-error`. **RMSE ≈ 0.693** summarizes all **84** errors from group 1 for depth 1.

In [9]:
round_errors <- validation_ready$log10_msc - round_predictions
sqrt(mean(round_errors^2))

[1] 0.692969

## 9.3 Combine the predictions from five rounds

Move the validation role from one group to the next. Each group takes one turn supplying records to check.

| Round | Group 1 | Group 2 | Group 3 | Group 4 | Group 5 |
|---|---|---|---|---|---|
| 1 | Check | Fit | Fit | Fit | Fit |
| 2 | Fit | Check | Fit | Fit | Fit |
| 3 | Fit | Fit | Check | Fit | Fit |
| 4 | Fit | Fit | Fit | Check | Fit |
| 5 | Fit | Fit | Fit | Fit | Check |

Run the supplied {span .script-block}`BLOCK · validate-depth`. It repeats the same fitting, predicting and checking steps for all five groups at each of the three depths.

In [10]:
# Repeat the illustrated fit/predict steps for all five folds and three depths.
tree_formula <- log10_msc ~ pasture_hours + milking_cows
oof <- matrix(NA_real_, nrow(train_survey), length(candidate_depths))
oof_baseline <- rep(NA_real_, nrow(train_survey))
fold_medians <- list()
validation_nodes <- list()
for (fold in 1:5) {
  fitting <- train_survey[row_fold != fold, ]
  validation <- train_survey[row_fold == fold, ]
  stopifnot(length(intersect(fitting$FarmID, validation$FarmID)) == 0)
  medians <- learn_medians(fitting)
  fold_medians[[fold]] <- data.frame(fold = fold, feature = tree_features, median = unname(medians))
  fitting <- fill_predictors(fitting, medians)
  validation <- fill_predictors(validation, medians)
  oof_baseline[row_fold == fold] <- mean(fitting$log10_msc)
  for (j in seq_along(candidate_depths)) {
    candidate <- rpart(tree_formula, data = fitting, method = "anova",
                       control = tree_control(candidate_depths[j]))
    candidate_nodes <- extract_tree_nodes(candidate)
    candidate_nodes$fold <- fold
    candidate_nodes$depth <- candidate_depths[j]
    validation_nodes[[length(validation_nodes) + 1L]] <- candidate_nodes
    oof[row_fold == fold, j] <- predict(candidate, newdata = validation)
  }
}
stopifnot(all(is.finite(oof)), all(is.finite(oof_baseline)))
observed <- train_survey$log10_msc
validation_scores <- data.frame(depth = candidate_depths,
  OOF_RMSE_log10 = vapply(seq_along(candidate_depths), function(j) sqrt(mean((observed - oof[, j])^2)), numeric(1)),
  OOF_MAE_log10 = vapply(seq_along(candidate_depths), function(j) mean(abs(observed - oof[, j])), numeric(1)))
baseline_validation <- data.frame(model = "Training-fold mean", depth = 0,
  OOF_RMSE_log10 = sqrt(mean((observed - oof_baseline)^2)), OOF_MAE_log10 = mean(abs(observed - oof_baseline)))

**Find the same record.** Run {span .script-block}`BLOCK · trace-validation-record` to look up the farm ID and sampling round you wrote down.

In [11]:
oof_table <- data.frame(FarmID = train_survey$FarmID, Sampling = train_survey$Sampling,
  fold = row_fold, observed_log10 = observed, baseline_prediction = oof_baseline,
  depth_1 = oof[, 1], depth_2 = oof[, 2], depth_3 = oof[, 3])
# Find the row with both the same farm ID and the same sampling round.
same_farm <- oof_table$FarmID == round_one_record$FarmID
same_round <- oof_table$Sampling == round_one_record$Sampling
traced_record <- oof_table[same_farm & same_round,
                           c("FarmID", "Sampling", "fold", "observed_log10", "depth_1")]
traced_record

,FarmID,Sampling,fold,observed_log10,depth_1
,<chr>,<chr>,<int>,<dbl>,<dbl>
3,R4,A,1,0.544068,0.6842116


Find **R4**, **A** and **fold 1** in the output. The `depth_1` value is about **0.6842**, the prediction you saw in Section 9.2. It came from the round in which group 1 was held out.

A prediction made while its group is held out is called an **out-of-fold prediction**, or **OOF prediction**. For each depth, the five rounds supply one such prediction for every training record. We combine all **420** errors into one RMSE per depth. Chapter 7 used the same calculation on the final test records. Here the errors come from the five groups inside the training set.

```{admonition} How the five groups make one RMSE
:class: note dropdown

For one depth, let $S_k$ be the sum of squared errors for the records held out in round $k$. Add the five sums, divide by the 420 records, and take the square root.

$$
\mathrm{RMSE}_{\mathrm{OOF}}=\sqrt{\frac{S_1+S_2+S_3+S_4+S_5}{420}}.
$$

Every record contributes one squared error, so a group with more records contributes more terms.
```

## 9.4 Choose a depth and check your guess

Choose the depth with the **smallest OOF RMSE**. If depths tie, choose the shallower one.

Run {span .script-block}`BLOCK · choose-depth`. The OOF RMSEs for depths 1, 2 and 3 are about **0.722**, **0.727** and **0.742**. Find the row with `selected = TRUE`. It marks **depth 1**.

In [12]:
# Treat RMSEs within 1e-12 of the lowest score as tied. Pick the shallowest tied tree.
chosen_depth <- min(validation_scores$depth[validation_scores$OOF_RMSE_log10 <= min(validation_scores$OOF_RMSE_log10) + 1e-12])
validation_scores$selected <- validation_scores$depth == chosen_depth
validation_scores
chosen_depth

depth,OOF_RMSE_log10,OOF_MAE_log10,selected
<int>,<dbl>,<dbl>,<lgl>
1,0.7219814,0.5646934,TRUE
2,0.7268655,0.5645752,FALSE
3,0.7423389,0.5842456,FALSE


[1] 1

**Return to your guess.** Run {span .script-block}`BLOCK · compare-training-and-validation-errors`. For each depth, read the error on the records used to fit the tree and the error on farms held out in validation.

In [13]:
# An illustration after selection: these fits do not alter the chosen depth.
training_medians <- learn_medians(train_survey)
train_filled <- fill_predictors(train_survey, training_medians)
validation_scores$training_RMSE_log10 <- vapply(candidate_depths, function(depth) {
  fitted <- rpart(tree_formula, data = train_filled, method = "anova", control = tree_control(depth))
  sqrt(mean((train_filled$log10_msc - predict(fitted, newdata = train_filled))^2))
}, numeric(1))
validation_scores[c("depth", "training_RMSE_log10", "OOF_RMSE_log10", "selected")]
baseline_validation

depth,training_RMSE_log10,OOF_RMSE_log10,selected
<int>,<dbl>,<dbl>,<lgl>
1,0.7014369,0.7219814,TRUE
2,0.6878793,0.7268655,FALSE
3,0.6767658,0.7423389,FALSE


model,depth,OOF_RMSE_log10,OOF_MAE_log10
<chr>,<dbl>,<dbl>,<dbl>
Training-fold mean,0,0.7206158,0.5574663


| Where we checked the predictions | What changed from depth 1 to depth 3? |
|---|---|
| The records used to fit the tree | Training RMSE fell from **0.701 to 0.677**. |
| Farms held out in each round | OOF RMSE rose from **0.722 to 0.742**. |

Which part matched your guess? Here, the deeper tree fit the known answers more closely, while the shallower tree predicted the held-out farms better.

**Compare with the baseline.** On those same validation records, the baseline’s OOF RMSE was **0.721**, slightly lower than the selected tree’s **0.722**. The baseline predicted each held-out group using the mean from that round’s fitting records.

## 9.5 Fit the final tree at the chosen depth

Each validation tree learned from four groups. We now know the depth and can fit the final tree using **all 420 records from the 81 training farms**. The comparison block in Section 9.4 already calculated the medians and prepared the inputs for this full training set. The final fit uses that prepared table to learn the branches and leaf means at depth 1.

**Check before running.** Which setting have we already chosen? What will the tree learn from the training data now?

Run {span .script-block}`BLOCK · fit-tree`. The result is stored as `final_tree`.

In [14]:
selected_settings <- fixed_tree_settings
selected_settings$maxdepth <- chosen_depth
final_tree <- rpart(log10_msc ~ pasture_hours + milking_cows,
                    data = train_filled, method = "anova", control = selected_settings)

## 9.6 Save the tree for Chapter 10

Run {span .script-block}`BLOCK · save-tree-fit`. In **Files → outputs → checkpoints**, find **`tree-fit.rds`**. It contains the final tree and the training medians used to prepare inputs.

In [15]:
write.csv(fold_table, "outputs/tree/training-farm-folds.csv", row.names = FALSE)
write.csv(do.call(rbind, fold_medians), "outputs/tree/fold-medians.csv", row.names = FALSE)
write.csv(do.call(rbind, validation_nodes), "outputs/tree/validation-nodes.csv", row.names = FALSE)
write.csv(validation_scores, "outputs/tree/validation-scores.csv", row.names = FALSE)
write.csv(baseline_validation, "outputs/tree/baseline-validation.csv", row.names = FALSE)
write.csv(oof_table, "outputs/tree/out-of-fold-predictions.csv", row.names = FALSE)
saveRDS(list(fit = final_tree, medians = training_medians, chosen_depth = chosen_depth,
             train = train_survey, test = tree_input$test, audit = tree_input$audit,
             validation = validation_scores, baseline_validation = baseline_validation),
        "outputs/checkpoints/tree-fit.rds")
cat("Saved tree-fit.rds; no tree test score has been calculated.\n")

Saved tree-fit.rds; no tree test score has been calculated.


The tables in **outputs → tree** let you check your work later.

| File | What you can check |
|---|---|
| `validation-scores.csv` | Which depth had the smallest validation RMSE |
| `training-farm-folds.csv` | Which group each farm belonged to |
| `out-of-fold-predictions.csv` | The predictions for each record when its group was held out |

**Next.** Chapter 10 reads the saved tree and applies it to the **21 final test farms**. [Read the tree, predict and compare](10-test-tree.ipynb).